In [1]:
import pandas as pd
import numpy as np
import json

with open("cluster_final.json", "r") as f:
    cluster_features = json.load(f)

cluster_flat = []
for cluster in cluster_features:
    cluster_flat += cluster
print("Number of features:", len(cluster_flat))

Number of features: 18814


In [15]:
import os

k = 5
seed = 3
# Folder containing your files
folder = f"synthetic_blocks_k{k}_{seed}"

# Expected range of i values
expected = set(range(0, len(cluster_features)))  # from 0 to 100 inclusive

# Get all files in folder
files = os.listdir(folder)

# Extract existing i values
existing = set()
for f in files:
    if f.startswith("synthetic_block_") and f.endswith(".csv"):
        try:
            i = int(f.replace("synthetic_block_", "").replace(".csv", ""))
            existing.add(i)
        except ValueError:
            pass  # skip malformed filenames

# Find missing indices
missing = sorted(expected - existing)

print("Missing indices:", missing)


Missing indices: []


In [3]:
or_data = pd.read_csv("../OriginalData/integrated_data_melanoma.csv")

In [16]:
concat_syns = []

for i in range(0, len(cluster_features)):
    block = pd.read_csv(f"synthetic_blocks_k{k}_{seed}/synthetic_block_{i}.csv", index_col =False)
    concat_syns.append(block)

synthetic_data = pd.concat(concat_syns, axis = 1)

original_order = or_data.columns.tolist()[1:]
ordred_synthetic_data = synthetic_data[original_order]

In [17]:
import sys
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)
from SynOmics.processing.metadata import MetaData
from SynOmics.processing.postprocessing import anonymize_ids

anonymized_df = anonymize_ids(
    ids = or_data["Patient"].values.tolist(),
    synthetic_data = ordred_synthetic_data,
    output_path = f"synthetic_blocks_k{k}_{seed}"
)
anonymized_df.to_csv(f"avatarsk{k}_{seed}.csv", index = False)